# 4편 가격을 바꾸면 이익이 움직인다
**쉽게 따라 해 보면서 배우는 파이썬**

위에서부터 한 셀씩 `Shift + Enter`. 설명은 사이트의 4편 페이지에 있다.

## 4-1 데이터 큐브: 월 × 요일 × 고객 유형

In [ ]:
# 1편 실습 파일 묶음(zip)을 다시 올리고 푼다
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall()
print(sorted(f for f in os.listdir() if f.startswith("koi_")))

In [ ]:
import pandas as pd
enc = "utf-8-sig"
sales = pd.read_csv("koi_sales.csv", encoding=enc, parse_dates=["date"]).drop_duplicates(subset="receipt_id")
sales["net"] = sales["qty"] * sales["price"] - sales["discount"]
sales["month"] = sales["date"].dt.month
sales["weekday"] = sales["date"].dt.day_name().str[:3]
sales["customer"] = sales["customer"].fillna("Unknown")      # 빈칸은 '모름'으로
cube = sales.groupby(["month", "weekday", "customer"])["net"].sum()
print("cells:", len(cube), "= 12 months x 7 days x", sales["customer"].nunique(), "types")
print(cube.head(8))

## 4-2 한 면 잘라 보기 · CH-14

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
regular = cube.xs("Regular", level="customer").unstack("weekday")[days] / 1e4    # 만원
print(regular.round(0))
share = cube.xs("Regular", level="customer").sum() / cube.sum()
print("regular share of sales:", round(share * 100, 1), "%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.heatmap(regular, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax)
ax.set(title="CH-14 Regular customers' net sales (KRW 10k)", xlabel="", ylabel="Month")
plt.show()

## 4-3 3D 산점, 정말 읽히나 · CH-15

In [ ]:
import numpy as np
c = cube.reset_index()
c["d"] = c["weekday"].map({d: i for i, d in enumerate(days)})
types = ["Regular", "Walk-in", "Tourist", "Unknown"]
c["t"] = c["customer"].map({t: i for i, t in enumerate(types)})
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
p = ax.scatter(c["month"], c["d"], c["t"], s=c["net"] / 2e4, c=c["net"] / 1e4, cmap="viridis")
ax.set(xlabel="Month", ylabel="Weekday", title="CH-15 Koi sales cube in 3D",
       yticks=range(7), yticklabels=days, zticks=range(4), zticklabels=types)
fig.colorbar(p, shrink=0.6, pad=0.15, label="KRW 10k")
plt.show()

In [ ]:
top = cube.idxmax()
print("largest cell:", top[0], top[1], top[2], "/", round(cube.max() / 1e4), "(KRW 10k)")

## 4-4 가정은 사전에, 계산은 격자로

In [ ]:
# 커피(아메리카노 · 라테) 한 달 잔 수: 2025년 실제 평균
coffee = sales[sales["menu"].isin(["Americano", "Latte"])]
base_cups = round(coffee.groupby("month")["qty"].sum().mean())
assume = {
    "base_cups": base_cups,   # 지금 가격일 때 한 달 잔 수
    "base_price": 4500,       # 지금 판매가 (원)
    "elasticity": -1.0,       # 값이 1% 오르면 잔 수가 1% 준다고 가정
    "grams": 18,              # 한 잔에 드는 원두 (g)
    "usdkrw": 1470,           # 환율 (3편, 12월 수준)
    "other": 600,             # 컵 · 우유 · 시럽 등 한 잔 기타 비용 (원)
    "fixed": 3_600_000,       # 커피에 나눠 준 한 달 고정비 (원)
}
def profit(price, bean_usd, a):
    """한 달 커피 이익(원). bean_usd = 원두 1kg 달러 가격"""
    cups = a["base_cups"] * (price / a["base_price"]) ** a["elasticity"]
    bean = bean_usd * a["usdkrw"] * a["grams"] / 1000     # 한 잔 원두값(원)
    return cups * (price - bean - a["other"]) - a["fixed"]
print(assume)
print("now ($14/kg, 4,500원):", round(profit(4500, 14, assume)))

In [ ]:
prices = np.linspace(3000, 6000, 61)      # 50원 간격
beans  = np.linspace(8, 30, 45)           # 0.5달러 간격
P, B = np.meshgrid(prices, beans)
Z = profit(P, B, assume) / 1e4           # 만원
print("grid:", Z.shape, "/ max", round(Z.max()), "/ min", round(Z.min()))

## 4-5 판매가 × 원두 단가 이익 등고선 · CH-16

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
cf = ax.contourf(P, B, Z, levels=20, cmap="RdYlGn")
zero = ax.contour(P, B, Z, levels=[0], colors="black", linewidths=2)
ax.clabel(zero, fmt="break-even")
ax.plot(4500, 14, "w*", markersize=14, markeredgecolor="black")      # 지금 위치
ax.set(title="CH-16 Koi coffee monthly profit (KRW 10k)", xlabel="Price (KRW)", ylabel="Beans (USD per kg)")
fig.colorbar(cf, label="KRW 10k")
plt.show()

In [ ]:
def breakeven(bean_usd, a):
    ok = prices[profit(prices, bean_usd, a) >= 0]
    return int(ok.min()) if len(ok) else None
for b in [10, 14, 20, 26]:
    print(f"beans ${b}/kg -> break-even price {breakeven(b, assume)}")

## 4-6 가정 바꾸기: 환율이 오르면

In [ ]:
hi = dict(assume)                       # 원래 가정은 그대로 두고 복사
hi["usdkrw"] = round(assume["usdkrw"] * 1.10)
Z2 = profit(P, B, hi) / 1e4
for b in [14, 20]:
    print(f"beans ${b}: break-even {breakeven(b, assume)} -> {breakeven(b, hi)}")
print("now profit:", round(profit(4500, 14, assume)), "->", round(profit(4500, 14, hi)))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
c1 = ax.contour(P, B, Z, levels=[0], colors="black", linewidths=2)
c2 = ax.contour(P, B, Z2, levels=[0], colors="#C9382A", linewidths=2, linestyles="--")
ax.clabel(c1, fmt=f"USD/KRW {assume['usdkrw']}"); ax.clabel(c2, fmt=f"USD/KRW {hi['usdkrw']}")
ax.plot(4500, 14, "k*", markersize=12)
ax.set(title="Break-even line moves when USD/KRW rises 10%", xlabel="Price (KRW)", ylabel="Beans (USD per kg)")
plt.show()

## 4-7 3D 곡면 vs 2D 등고선

In [ ]:
fig = plt.figure(figsize=(11, 4.5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(P, B, Z, cmap="RdYlGn")
ax1.set(title="3D surface", xlabel="Price", ylabel="Beans $", zlabel="Profit")
ax2 = fig.add_subplot(1, 2, 2)
ax2.contourf(P, B, Z, levels=20, cmap="RdYlGn")
ax2.contour(P, B, Z, levels=[0], colors="black", linewidths=2)
ax2.set(title="2D contour", xlabel="Price (KRW)", ylabel="Beans (USD per kg)")
fig.subplots_adjust(wspace=0.35)
plt.show()

## 4-8 보고서용으로 저장하고 내려받기

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
cf = ax.contourf(P, B, Z, levels=20, cmap="RdYlGn")
ax.contour(P, B, Z, levels=[0], colors="black", linewidths=2)
ax.plot(4500, 14, "w*", markersize=14, markeredgecolor="black")
ax.set(title="Koi coffee: price x bean cost -> monthly profit", xlabel="Price (KRW)", ylabel="Beans (USD per kg)")
fig.colorbar(cf, label="KRW 10k")
fig.savefig("koi_breakeven.png", dpi=200, bbox_inches="tight")    # 보고서용 고해상도
print("saved:", round(os.path.getsize("koi_breakeven.png") / 1024), "KB")
plt.close(fig)

In [ ]:
# Colab 에서만 동작한다. 브라우저가 파일을 내려받는다
try:
    from google.colab import files
    files.download("koi_breakeven.png")
except ImportError:
    print("Colab 이 아니면 왼쪽 파일 목록이나 작업 폴더에서 직접 연다")